In [4]:
# Shared setup: CSV + PostgreSQL (Chroma has its own cells below).
# pip install pandas sqlalchemy psycopg2-binary python-dotenv
import os
import re
from pathlib import Path
from urllib.parse import quote_plus

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.types import BigInteger, Float, String, Text

_here = Path.cwd().resolve()
for _env in [_here / ".env", _here.parent / ".env"]:
    if _env.is_file():
        load_dotenv(_env)
        break
else:
    load_dotenv()

# PostgreSQL — set in .env: POSTGRES_USER, POSTGRES_PASSWORD, POSTGRES_HOST, POSTGRES_PORT, POSTGRES_DB
POSTGRES_USER = os.getenv("POSTGRES_USER", "postgres")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD", "root")
POSTGRES_HOST = os.getenv("POSTGRES_HOST", "localhost")
POSTGRES_PORT = os.getenv("POSTGRES_PORT", "5432")
POSTGRES_DB = os.getenv("POSTGRES_DB", "reddit_etl")
PG_TABLE = os.getenv("POSTGRES_TABLE", "reddit_posts_mental_emotion")
POSTGRES_ADMIN_DB = os.getenv("POSTGRES_ADMIN_DB", "postgres")

In [5]:
_csv = Path("reddit_posts_categorized_mental_status_with_emotion.csv")
if not _csv.is_file():
    _csv = Path("reddit_data_proces") / "reddit_posts_categorized_mental_status_with_emotion.csv"

df = pd.read_csv(_csv, encoding="utf-8")
if "data_source" not in df.columns:
    df["data_source"] = "R-OpiatesRecovery"

print(f"Loaded {len(df):,} rows from {_csv.resolve()}")
df[["title", "selftext", "comment_body"]].head(2)

Loaded 2,134 rows from D:\Sajjad-Workspace\SocialMediaData_ETL\reddit_data_proces\reddit_posts_categorized_mental_status_with_emotion.csv


,title,selftext,comment_body
0,Sat/Sun March 21/22 check in,"Hey guys, hope your weekend’s going well! It’s...",I really appreciate the weekends. Went to yoga...
1,❣️Reminder to keep us safe:,"Over the last month, I’ve received a few repor...",I really really wish Reddit would allow us to ...


In [6]:
# ---- PostgreSQL: create DB if missing, then load dataframe ----
def _pg_url(db_name: str) -> str:
    user = quote_plus(POSTGRES_USER)
    pwd = quote_plus(POSTGRES_PASSWORD or "")
    return f"postgresql+psycopg2://{user}:{pwd}@{POSTGRES_HOST}:{POSTGRES_PORT}/{db_name}"


if not re.fullmatch(r"[A-Za-z0-9_]+", POSTGRES_DB):
    raise ValueError("POSTGRES_DB must contain only letters, digits, and underscores")

engine_admin = create_engine(_pg_url(POSTGRES_ADMIN_DB), isolation_level="AUTOCOMMIT")
with engine_admin.connect() as conn:
    exists = conn.execute(
        text("SELECT 1 FROM pg_database WHERE datname = :d"),
        {"d": POSTGRES_DB},
    ).scalar()
    if not exists:
        conn.execute(text(f'CREATE DATABASE "{POSTGRES_DB}"'))
        print(f"Created database: {POSTGRES_DB}")
engine_admin.dispose()

engine = create_engine(_pg_url(POSTGRES_DB))

dtype_map = {
    "title": Text(),
    "selftext": Text(),
    "comment_body": Text(),
    "score": BigInteger(),
    "comment_score": Float(),
    "post_word_count": BigInteger(),
    "post_size_category": String(64),
    "mental_health_class": String(128),
    "post_emotion": String(64),
    "comment_emotion": String(64),
    "data_source": String(128),
}

df.to_sql(
    PG_TABLE,
    engine,
    if_exists="replace",
    index=False,
    dtype=dtype_map,
    chunksize=500,
    method="multi",
)
print(f"Wrote {len(df):,} rows to {POSTGRES_DB} public.{PG_TABLE}")

with engine.connect() as c:
    n = c.execute(text(f'SELECT COUNT(*) FROM "{PG_TABLE}"')).scalar()
print("PostgreSQL row count:", n)

Created database: reddit_etl
Wrote 2,134 rows to reddit_etl public.reddit_posts_mental_emotion
PostgreSQL row count: 2134


In [7]:
# ---- Chroma DB: imports & paths ----
# pip install chromadb sentence-transformers
# Requires earlier cells: _here (cell 0), df (cell 1).
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

EMBED_MODEL = "sentence-transformers/all-roberta-large-v1"
CHROMA_DIR = _here / "chroma_reddit_mental_emotion"
COLLECTION_NAME = "reddit_post_comment_roberta_large_v1"

In [8]:
# ---- Chroma DB: helpers for documents & metadata ----
def _text_or_placeholder(x) -> str:
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return "[no text]"
    s = str(x).strip()
    return s if s else "[no text]"


def _meta_scalar(v):
    """Chroma metadata: str, int, float, or bool only (no NaN)."""
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ""
    if isinstance(v, (bool,)):
        return v
    if isinstance(v, (int,)) and not isinstance(v, bool):
        return int(v)
    if isinstance(v, float):
        return float(v)
    return str(v)[:500]

In [9]:
# ---- Chroma DB: persistent client, collection, batch upsert (selftext + comment_body) ----
embed_fn = SentenceTransformerEmbeddingFunction(model_name=EMBED_MODEL)
client = chromadb.PersistentClient(path=str(CHROMA_DIR))

collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embed_fn,
)

BATCH = 64
ids_batch, docs_batch, metas_batch = [], [], []
total = 0

for i in range(len(df)):
    row = df.iloc[i]
    base = {
        "csv_row_index": int(i),
        "title": _meta_scalar(row.get("title")),
        "score": _meta_scalar(row.get("score")),
        "comment_score": _meta_scalar(row.get("comment_score")),
        "post_word_count": _meta_scalar(row.get("post_word_count")),
        "post_size_category": _meta_scalar(row.get("post_size_category")),
        "mental_health_class": _meta_scalar(row.get("mental_health_class")),
        "post_emotion": _meta_scalar(row.get("post_emotion")),
        "comment_emotion": _meta_scalar(row.get("comment_emotion")),
        "data_source": _meta_scalar(row.get("data_source")),
    }

    post_doc = _text_or_placeholder(row.get("selftext"))
    ids_batch.append(f"r{i}_post")
    docs_batch.append(post_doc)
    metas_batch.append({**base, "content_type": "post", "embedded_field": "selftext"})

    com_doc = _text_or_placeholder(row.get("comment_body"))
    ids_batch.append(f"r{i}_comment")
    docs_batch.append(com_doc)
    metas_batch.append({**base, "content_type": "comment", "embedded_field": "comment_body"})

    if len(ids_batch) >= BATCH:
        collection.upsert(ids=ids_batch, documents=docs_batch, metadatas=metas_batch)
        total += len(ids_batch)
        ids_batch, docs_batch, metas_batch = [], [], []

if ids_batch:
    collection.upsert(ids=ids_batch, documents=docs_batch, metadatas=metas_batch)
    total += len(ids_batch)

print(f"Upserted {total:,} vectors ({len(df):,} rows × post + comment) into collection '{COLLECTION_NAME}'")
print("count:", collection.count())

c:\Users\nipua\AppData\Local\anaconda3\envs\py312_gpu_2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Upserted 4,268 vectors (2,134 rows × post + comment) into collection 'reddit_post_comment_roberta_large_v1'
count: 4268


In [10]:
# ---- Chroma DB: sanity query (semantic search) ----
_q = "withdrawal symptoms and tapering"
hits = collection.query(query_texts=[_q], n_results=5)
for j, (_id, doc, meta, dist) in enumerate(
    zip(hits["ids"][0], hits["documents"][0], hits["metadatas"][0], hits["distances"][0])
):
    print(j + 1, _id, meta.get("content_type"), "distance=", round(dist, 4))
    print((doc or "")[:200], "...\n")

1 r1750_comment comment distance= 0.3366
Yes, this will reduce the intensity of the withdrawal, by extending the duration. Even a rapid taper is better than just going full cold turkey.

I think your plan seems reasonable. If after stepping  ...

2 r1194_post post distance= 0.3428
Many of us have differrent signs of withdrawal approaching. Lets hear what you go through and what starts first for you? 

For me, personally i get an extreme sense of anxiety and a feeling of "doom". ...

3 r1195_post post distance= 0.3428
Many of us have differrent signs of withdrawal approaching. Lets hear what you go through and what starts first for you? 

For me, personally i get an extreme sense of anxiety and a feeling of "doom". ...

4 r1196_post post distance= 0.3428
Many of us have differrent signs of withdrawal approaching. Lets hear what you go through and what starts first for you? 

For me, personally i get an extreme sense of anxiety and a feeling of "doom". ...

5 r1441_comment comment distan